# 0825_peace_018_type_expert_fold_ensemble_sensor_anomaly

005 Fold 앙상블에 015의 Train-only 타입별 센서 이상도 피처 5개만 추가합니다. 모델·분할·앙상블·임계값 규칙은 고정하며 80~100% Test는 추론하지 않습니다.

## 1. 설정과 실행 로그

In [1]:
import gc
import hashlib
import json
import logging
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import sklearn
import xgboost
from IPython.display import display
from sklearn.compose import ColumnTransformer
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.preprocessing import OneHotEncoder
from xgboost import XGBClassifier

EXPERIMENT_ID = "0825_peace_018_type_expert_fold_ensemble_sensor_anomaly"
RANDOM_STATE = 42
TARGET = "class"
TIME_COLUMN = "timestamp"
TYPE_COLUMN = "inspection_type"
RECORD_ID = "record_id"
DECISION_THRESHOLD = 0.5
MIN_RECALL = 0.99
TRAIN_END_FRACTION = 0.70
VALIDATION_END_FRACTION = 0.80

XGB_PARAMS = {
    "objective": "binary:logistic",
    "eval_metric": "aucpr",
    "tree_method": "hist",
    "n_estimators": 400,
    "learning_rate": 0.05,
    "max_depth": 5,
    "min_child_weight": 10,
    "subsample": 0.8,
    "colsample_bytree": 0.8,
    "reg_alpha": 0.1,
    "reg_lambda": 5.0,
    "max_delta_step": 1.0,
    "random_state": RANDOM_STATE,
    "n_jobs": -1,
    "verbosity": 0,
}


def find_repo_root() -> Path:
    for candidate in [Path.cwd(), *Path.cwd().parents]:
        if (candidate / "AGENTS.md").exists() and (candidate / "notebooks").is_dir():
            return candidate.resolve()
    raise FileNotFoundError("AGENTS.md가 있는 저장소 루트를 찾지 못했습니다.")


def sha256_file(path: Path, chunk_size: int = 1024 * 1024) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as stream:
        for chunk in iter(lambda: stream.read(chunk_size), b""):
            digest.update(chunk)
    return digest.hexdigest()


def find_data_pair(repo_root: Path) -> tuple[Path, Path]:
    candidates = [
        repo_root / "data" / "raw",
        repo_root.parent,
        Path.cwd(),
        Path.cwd().parent,
        Path.cwd().parent.parent,
    ]
    checked = set()
    for directory in candidates:
        resolved = directory.resolve()
        if resolved in checked:
            continue
        checked.add(resolved)
        data_path = resolved / "dataset.csv"
        mapping_path = resolved / "mapping.json"
        if data_path.exists() and mapping_path.exists():
            return data_path, mapping_path
    raise FileNotFoundError("dataset.csv와 mapping.json 쌍을 찾지 못했습니다.")


REPO_ROOT = find_repo_root()
DATA_PATH, MAPPING_PATH = find_data_pair(REPO_ROOT)
LOG_DIR = REPO_ROOT / "docs" / "peace"
LOG_DIR.mkdir(parents=True, exist_ok=True)
LOG_PATH = LOG_DIR / f"{EXPERIMENT_ID}.log"

logger = logging.getLogger(EXPERIMENT_ID)
logger.setLevel(logging.INFO)
logger.handlers.clear()
formatter = logging.Formatter("%(asctime)s | %(levelname)s | %(message)s")
file_handler = logging.FileHandler(LOG_PATH, mode="w", encoding="utf-8")
file_handler.setFormatter(formatter)
stream_handler = logging.StreamHandler(sys.stdout)
stream_handler.setFormatter(formatter)
logger.addHandler(file_handler)
logger.addHandler(stream_handler)
logger.propagate = False

DATA_SHA256_BEFORE = sha256_file(DATA_PATH)
MAPPING_SHA256_BEFORE = sha256_file(MAPPING_PATH)
logger.info("experiment=%s", EXPERIMENT_ID)
logger.info(
    "random_state=%d baseline_threshold=%.2f min_recall=%.2f",
    RANDOM_STATE,
    DECISION_THRESHOLD,
    MIN_RECALL,
)
logger.info("data_file=%s sha256=%s", DATA_PATH.name, DATA_SHA256_BEFORE)
logger.info("mapping_file=%s sha256=%s", MAPPING_PATH.name, MAPPING_SHA256_BEFORE)
logger.info(
    "versions python=%s pandas=%s sklearn=%s xgboost=%s",
    sys.version.split()[0], pd.__version__, sklearn.__version__, xgboost.__version__
)
logger.info("log_file=docs/peace/%s", LOG_PATH.name)
print("log saved to:", LOG_PATH.relative_to(REPO_ROOT))


BASELINE_005_WALK = {
    "global_threshold": {
        "mean_pr_auc": 0.06666848491626425,
        "mean_recall": 0.9868421052631579,
        "min_recall": 0.9605263157894737,
        "recall_99_folds": 2,
        "mean_false_call_reduction": 0.17635946579785103,
    }
}
BASELINE_005_VALIDATION = {
    "global_threshold": {
        "pr_auc": 0.3826504981185914,
        "recall": 0.9915966386554622,
        "false_call_reduction": 0.6110742174082301,
        "tp": 354,
        "fn": 3,
        "fp": 16984,
        "tn": 26685,
    }
}


def to_builtin_dict(value):
    if isinstance(value, dict):
        return {str(key): to_builtin_dict(item) for key, item in value.items()}
    if isinstance(value, list):
        return [to_builtin_dict(item) for item in value]
    if isinstance(value, (np.integer,)):
        return int(value)
    if isinstance(value, (np.floating,)):
        return float(value)
    return value


2026-08-25 20:07:37,440 | INFO | experiment=0825_peace_018_type_expert_fold_ensemble_sensor_anomaly


2026-08-25 20:07:37,441 | INFO | random_state=42 baseline_threshold=0.50 min_recall=0.99


2026-08-25 20:07:37,441 | INFO | data_file=dataset.csv sha256=53e8568743216d556856ed69b388f6750fbfa0b8c59ad31f970515ac9eb10e62


2026-08-25 20:07:37,442 | INFO | mapping_file=mapping.json sha256=3b20f440b6d9ed0baefa662e1a6f03688befbe0f28341a3b54655d3058c6e486


2026-08-25 20:07:37,442 | INFO | versions python=3.12.7 pandas=2.2.2 sklearn=1.5.1 xgboost=3.4.1


2026-08-25 20:07:37,442 | INFO | log_file=docs/peace/0825_peace_018_type_expert_fold_ensemble_sensor_anomaly.log


log saved to: docs/peace/0825_peace_018_type_expert_fold_ensemble_sensor_anomaly.log


## 2. 원본 데이터, 매핑과 타입별 유효 피처

In [2]:
raw_df = pd.read_csv(DATA_PATH, low_memory=False)
source_index_column = raw_df.columns[0]
if source_index_column.startswith("Unnamed:") or source_index_column == "":
    raw_df = raw_df.rename(columns={source_index_column: RECORD_ID})
elif source_index_column != RECORD_ID:
    raise ValueError(f"예상하지 못한 첫 번째 컬럼: {source_index_column}")

with MAPPING_PATH.open(encoding="utf-8") as stream:
    feature_mapping = json.load(stream)

required_columns = {RECORD_ID, TIME_COLUMN, TYPE_COLUMN, TARGET}
missing_required = required_columns - set(raw_df.columns)
assert not missing_required, f"필수 컬럼 누락: {sorted(missing_required)}"
assert len(raw_df) == 440_274
assert raw_df[RECORD_ID].is_unique
assert set(raw_df[TARGET].unique()) == {0, 1}
assert raw_df[TARGET].value_counts().to_dict() == {0: 435_652, 1: 4_622}
assert set(raw_df[TYPE_COLUMN].unique()) == {0, 1, 2, 3, 4}
assert set(feature_mapping) == {"0", "1", "2", "3", "4"}

raw_df[TIME_COLUMN] = pd.to_datetime(raw_df[TIME_COLUMN], errors="raise", utc=True)
raw_df = raw_df.sort_values([TIME_COLUMN, RECORD_ID], kind="stable").reset_index(drop=True)
inspection_columns = [column for column in raw_df.columns if column.startswith("inspection_feat")]
mapped_union = set().union(*(set(columns) for columns in feature_mapping.values()))
assert len(inspection_columns) == 70
assert len(mapped_union) == 65
assert mapped_union <= set(inspection_columns)

numeric_inputs = raw_df.select_dtypes(include=[np.number]).drop(columns=[TARGET, RECORD_ID])
assert np.isfinite(numeric_inputs.to_numpy()).all()

data_summary = pd.Series(
    {
        "rows": len(raw_df),
        "columns": raw_df.shape[1],
        "false_call_0": int((raw_df[TARGET] == 0).sum()),
        "real_defect_1": int((raw_df[TARGET] == 1).sum()),
        "real_defect_rate_pct": raw_df[TARGET].mean() * 100,
        "inspection_types": raw_df[TYPE_COLUMN].nunique(),
        "inspection_features": len(inspection_columns),
        "mapped_feature_union": len(mapped_union),
        "timestamp_start": raw_df[TIME_COLUMN].min(),
        "timestamp_end": raw_df[TIME_COLUMN].max(),
    },
    name="raw_data",
)
display(data_summary)
logger.info(
    "data_verified rows=%d columns=%d class_0=%d class_1=%d",
    len(raw_df), raw_df.shape[1], int((raw_df[TARGET] == 0).sum()), int((raw_df[TARGET] == 1).sum())
)


rows                                       440274
columns                                        78
false_call_0                               435652
real_defect_1                                4622
real_defect_rate_pct                     1.049801
inspection_types                                5
inspection_features                            70
mapped_feature_union                           65
timestamp_start         1970-06-23 03:58:55+00:00
timestamp_end           1970-11-02 14:21:28+00:00
Name: raw_data, dtype: object

2026-08-25 20:07:41,809 | INFO | data_verified rows=440274 columns=78 class_0=435652 class_1=4622


In [3]:
inspection_types = sorted(raw_df[TYPE_COLUMN].unique().tolist())
meta_columns = [column for column in raw_df.columns if column.startswith("meta_feat")]
feature_columns_by_type = {}
feature_rows = []

for inspection_type in inspection_types:
    mapped_columns = feature_mapping[str(inspection_type)]
    assert len(mapped_columns) == len(set(mapped_columns))
    assert set(mapped_columns) <= set(raw_df.columns)
    selected_columns = meta_columns + mapped_columns
    feature_columns_by_type[inspection_type] = selected_columns
    feature_rows.append(
        {
            "inspection_type": inspection_type,
            "meta_features": len(meta_columns),
            "mapped_inspection_features": len(mapped_columns),
            "total_model_features": len(selected_columns),
        }
    )

feature_summary = pd.DataFrame(feature_rows).set_index("inspection_type")
display(feature_summary)
logger.info("feature_mapping_verified=%s", feature_summary.to_dict(orient="index"))


mapped_feature_columns_by_type = {
    inspection_type: list(feature_mapping[str(inspection_type)])
    for inspection_type in inspection_types
}


,meta_features,mapped_inspection_features,total_model_features
inspection_type,,,
0,4,44,48
1,4,52,56
2,4,65,69
3,4,65,69
4,4,21,25


2026-08-25 20:07:41,837 | INFO | feature_mapping_verified={0: {'meta_features': 4, 'mapped_inspection_features': 44, 'total_model_features': 48}, 1: {'meta_features': 4, 'mapped_inspection_features': 52, 'total_model_features': 56}, 2: {'meta_features': 4, 'mapped_inspection_features': 65, 'total_model_features': 69}, 3: {'meta_features': 4, 'mapped_inspection_features': 65, 'total_model_features': 69}, 4: {'meta_features': 4, 'mapped_inspection_features': 21, 'total_model_features': 25}}


## 3. 시간순 분할과 Test holdout 보호

In [4]:
timestamp_group_sizes = raw_df.groupby(TIME_COLUMN, sort=True).size()
cumulative_rows = timestamp_group_sizes.cumsum().to_numpy()
timestamp_index = timestamp_group_sizes.index


def boundary_at(fraction: float):
    position = int(np.searchsorted(cumulative_rows, len(raw_df) * fraction, side="left"))
    return timestamp_index[position]


train_end_time = boundary_at(TRAIN_END_FRACTION)
validation_end_time = boundary_at(VALIDATION_END_FRACTION)
train_mask = raw_df[TIME_COLUMN] <= train_end_time
validation_mask = (
    (raw_df[TIME_COLUMN] > train_end_time)
    & (raw_df[TIME_COLUMN] <= validation_end_time)
)
test_mask = raw_df[TIME_COLUMN] > validation_end_time

train_df = raw_df.loc[train_mask]
validation_df = raw_df.loc[validation_mask]
test_holdout_df = raw_df.loc[test_mask]
assert train_df[TIME_COLUMN].max() < validation_df[TIME_COLUMN].min()
assert set(train_df[TIME_COLUMN]).isdisjoint(set(validation_df[TIME_COLUMN]))
assert validation_df[TIME_COLUMN].max() < test_holdout_df[TIME_COLUMN].min()
assert set(validation_df[TIME_COLUMN]).isdisjoint(set(test_holdout_df[TIME_COLUMN]))
assert int(train_mask.sum() + validation_mask.sum() + test_mask.sum()) == len(raw_df)

split_summary = pd.DataFrame(
    [
        {
            "split": "train",
            "rows": len(train_df),
            "positive_samples": int(train_df[TARGET].sum()),
            "positive_rate_pct": train_df[TARGET].mean() * 100,
            "timestamp_groups": train_df[TIME_COLUMN].nunique(),
            "start_time": train_df[TIME_COLUMN].min(),
            "end_time": train_df[TIME_COLUMN].max(),
        },
        {
            "split": "validation",
            "rows": len(validation_df),
            "positive_samples": int(validation_df[TARGET].sum()),
            "positive_rate_pct": validation_df[TARGET].mean() * 100,
            "timestamp_groups": validation_df[TIME_COLUMN].nunique(),
            "start_time": validation_df[TIME_COLUMN].min(),
            "end_time": validation_df[TIME_COLUMN].max(),
        },
        {
            "split": "test_holdout_unused",
            "rows": len(test_holdout_df),
            "positive_samples": int(test_holdout_df[TARGET].sum()),
            "positive_rate_pct": test_holdout_df[TARGET].mean() * 100,
            "timestamp_groups": test_holdout_df[TIME_COLUMN].nunique(),
            "start_time": test_holdout_df[TIME_COLUMN].min(),
            "end_time": test_holdout_df[TIME_COLUMN].max(),
        },
    ]
).set_index("split")
display(split_summary)
evaluation_policy = pd.Series(
    {
        "test_holdout_used_for_selection": False,
        "test_holdout_predicted": False,
    },
    name="evaluation_policy",
)
display(evaluation_policy)
logger.info("split_summary=%s", split_summary.reset_index().to_dict(orient="records"))
logger.info("test_policy selection=False predicted=False")


,rows,positive_samples,positive_rate_pct,timestamp_groups,start_time,end_time
split,,,,,,
train,308196,1940,0.629470,29249,1970-06-23 03:58:55+00:00,1970-10-05 00:29:59+00:00
validation,44026,357,0.810884,3400,1970-10-05 00:30:30+00:00,1970-10-13 16:54:14+00:00
test_holdout_unused,88052,2325,2.640485,7093,1970-10-13 16:54:52+00:00,1970-11-02 14:21:28+00:00


test_holdout_used_for_selection    False
test_holdout_predicted             False
Name: evaluation_policy, dtype: bool

2026-08-25 20:07:42,274 | INFO | split_summary=[{'split': 'train', 'rows': 308196, 'positive_samples': 1940, 'positive_rate_pct': 0.6294695583330089, 'timestamp_groups': 29249, 'start_time': Timestamp('1970-06-23 03:58:55+0000', tz='UTC'), 'end_time': Timestamp('1970-10-05 00:29:59+0000', tz='UTC')}, {'split': 'validation', 'rows': 44026, 'positive_samples': 357, 'positive_rate_pct': 0.8108844773542907, 'timestamp_groups': 3400, 'start_time': Timestamp('1970-10-05 00:30:30+0000', tz='UTC'), 'end_time': Timestamp('1970-10-13 16:54:14+0000', tz='UTC')}, {'split': 'test_holdout_unused', 'rows': 88052, 'positive_samples': 2325, 'positive_rate_pct': 2.640485167855358, 'timestamp_groups': 7093, 'start_time': Timestamp('1970-10-13 16:54:52+0000', tz='UTC'), 'end_time': Timestamp('1970-11-02 14:21:28+0000', tz='UTC')}]


2026-08-25 20:07:42,275 | INFO | test_policy selection=False predicted=False


## 4. 평가 함수와 Walk-forward 구간

In [5]:
def evaluate_predictions(y_true, prediction, probability):
    y_true = np.asarray(y_true, dtype=np.int8)
    prediction = np.asarray(prediction, dtype=np.int8)
    probability = np.asarray(probability, dtype=np.float64)
    tn, fp, fn, tp = confusion_matrix(y_true, prediction, labels=[0, 1]).ravel()
    has_both_classes = np.unique(y_true).size == 2
    has_positive = (tp + fn) > 0
    return {
        "rows": len(y_true),
        "positive_samples": int(y_true.sum()),
        "tn": int(tn),
        "fp": int(fp),
        "fn": int(fn),
        "tp": int(tp),
        "accuracy": accuracy_score(y_true, prediction),
        "precision": precision_score(y_true, prediction, zero_division=0),
        "recall": recall_score(y_true, prediction, zero_division=0) if has_positive else np.nan,
        "false_call_reduction": tn / (tn + fp) if (tn + fp) else np.nan,
        "f1": f1_score(y_true, prediction, zero_division=0) if has_positive else np.nan,
        "roc_auc": roc_auc_score(y_true, probability) if has_both_classes else np.nan,
        "pr_auc": average_precision_score(y_true, probability) if has_both_classes else np.nan,
    }


def evaluate_probabilities(y_true, probability, threshold=DECISION_THRESHOLD):
    probability = np.asarray(probability, dtype=np.float64)
    prediction = (probability >= threshold).astype(np.int8)
    return evaluate_predictions(y_true, prediction, probability)


def select_threshold(y_true, probability, min_recall=MIN_RECALL):
    """Recall 제약을 만족하며 False Call Reduction이 최대인 threshold를 선택한다."""
    y_true = np.asarray(y_true, dtype=np.int8)
    probability = np.asarray(probability, dtype=np.float64)
    if np.unique(y_true).size != 2:
        raise ValueError("임계값 선택에는 positive와 negative가 모두 필요합니다.")

    order = np.argsort(-probability, kind="stable")
    sorted_probability = probability[order]
    sorted_target = y_true[order]
    cumulative_tp = np.cumsum(sorted_target == 1)
    cumulative_fp = np.cumsum(sorted_target == 0)
    group_ends = np.flatnonzero(
        np.r_[sorted_probability[:-1] != sorted_probability[1:], True]
    )

    thresholds = sorted_probability[group_ends]
    tp = cumulative_tp[group_ends]
    fp = cumulative_fp[group_ends]
    total_positive = int((y_true == 1).sum())
    total_negative = int((y_true == 0).sum())
    recall = tp / total_positive
    false_call_reduction = 1.0 - (fp / total_negative)
    feasible = np.flatnonzero(recall >= min_recall)
    if feasible.size == 0:
        raise RuntimeError(f"Recall {min_recall:.2%} 조건을 만족하는 threshold가 없습니다.")

    best_local = np.lexsort(
        (thresholds[feasible], recall[feasible], false_call_reduction[feasible])
    )[-1]
    best = feasible[best_local]
    selected_threshold = float(thresholds[best])
    metrics = evaluate_probabilities(y_true, probability, selected_threshold)
    return {"threshold": selected_threshold, "min_recall": min_recall, **metrics}


# 최적화 구현이 작은 합성 예제의 완전 탐색과 같은 결과인지 검증한다.
_test_y = np.array([1, 0, 1, 0, 1, 0], dtype=np.int8)
_test_probability = np.array([0.9, 0.8, 0.7, 0.6, 0.4, 0.2])
_optimized = select_threshold(_test_y, _test_probability, min_recall=2 / 3)
_reference_rows = []
for _threshold in np.unique(_test_probability):
    _metrics = evaluate_probabilities(_test_y, _test_probability, _threshold)
    if _metrics["recall"] >= 2 / 3:
        _reference_rows.append((_metrics["false_call_reduction"], _metrics["recall"], _threshold))
_reference = max(_reference_rows)
assert np.isclose(_optimized["threshold"], _reference[2])
logger.info("threshold_selector_unit_test=PASS")

def make_preprocessor(feature_columns):
    categorical = [column for column in meta_columns if column in feature_columns]
    continuous = [column for column in feature_columns if column not in categorical]
    return ColumnTransformer(
        transformers=[
            (
                "categorical",
                OneHotEncoder(handle_unknown="ignore", dtype=np.float32),
                categorical,
            ),
            ("continuous", "passthrough", continuous),
        ],
        sparse_threshold=1.0,
        verbose_feature_names_out=True,
    )


def evaluate_calibration_and_future(calibration_frame, calibration_probability, evaluation_frame, evaluation_probability, stage_name):
    global_selection = select_threshold(calibration_frame[TARGET], calibration_probability, min_recall=MIN_RECALL)
    type_thresholds = {}
    type_prediction = pd.Series(np.nan, index=evaluation_frame.index, dtype="float64")
    threshold_rows = [{"stage": stage_name, "scope": "global", **global_selection}]
    type_rows = []
    for inspection_type in inspection_types:
        type_calibration = calibration_frame.loc[calibration_frame[TYPE_COLUMN] == inspection_type]
        selection = select_threshold(type_calibration[TARGET], calibration_probability.loc[type_calibration.index], min_recall=MIN_RECALL)
        type_thresholds[inspection_type] = selection["threshold"]
        threshold_rows.append({"stage": stage_name, "scope": f"type_{inspection_type}", **selection})
        type_evaluation = evaluation_frame.loc[evaluation_frame[TYPE_COLUMN] == inspection_type]
        type_probability = evaluation_probability.loc[type_evaluation.index]
        prediction = (type_probability >= selection["threshold"]).astype("int8")
        type_prediction.loc[type_evaluation.index] = prediction
        metrics = evaluate_predictions(type_evaluation[TARGET], prediction, type_probability)
        type_rows.append({"stage": stage_name, "inspection_type": inspection_type, "threshold": selection["threshold"], **metrics})
    strategy_metrics = {
        "fixed_0.5": evaluate_probabilities(evaluation_frame[TARGET], evaluation_probability, DECISION_THRESHOLD),
        "global_threshold": evaluate_probabilities(evaluation_frame[TARGET], evaluation_probability, global_selection["threshold"]),
        "type_specific_thresholds": evaluate_predictions(evaluation_frame[TARGET], type_prediction, evaluation_probability),
    }
    metric_rows = [{"stage": stage_name, "strategy": strategy, **metrics} for strategy, metrics in strategy_metrics.items()]
    return {"global_selection": global_selection, "type_thresholds": type_thresholds, "threshold_rows": threshold_rows, "type_rows": type_rows, "metric_rows": metric_rows}

2026-08-25 20:07:42,302 | INFO | threshold_selector_unit_test=PASS


In [6]:
WALK_FORWARD_SPECS = [
    {
        "fold": "fold_1",
        "train_start": 0.00,
        "train_end": 0.30,
        "calibration_start": 0.30,
        "calibration_end": 0.40,
        "evaluation_start": 0.40,
        "evaluation_end": 0.50,
    },
    {
        "fold": "fold_2",
        "train_start": 0.00,
        "train_end": 0.40,
        "calibration_start": 0.40,
        "calibration_end": 0.50,
        "evaluation_start": 0.50,
        "evaluation_end": 0.60,
    },
    {
        "fold": "fold_3",
        "train_start": 0.00,
        "train_end": 0.50,
        "calibration_start": 0.50,
        "calibration_end": 0.60,
        "evaluation_start": 0.60,
        "evaluation_end": 0.70,
    },
]

walk_forward_boundaries = {
    fraction: boundary_at(fraction)
    for fraction in [0.30, 0.40, 0.50, 0.60, 0.70]
}
walk_forward_segments = {}
walk_forward_split_rows = []

for spec in WALK_FORWARD_SPECS:
    fold_name = spec["fold"]
    train_end = walk_forward_boundaries[spec["train_end"]]
    calibration_start = walk_forward_boundaries[spec["calibration_start"]]
    calibration_end = walk_forward_boundaries[spec["calibration_end"]]
    evaluation_start = walk_forward_boundaries[spec["evaluation_start"]]
    evaluation_end = walk_forward_boundaries[spec["evaluation_end"]]

    segments = {
        "train": raw_df.loc[raw_df[TIME_COLUMN] <= train_end],
        "calibration": raw_df.loc[
            (raw_df[TIME_COLUMN] > calibration_start)
            & (raw_df[TIME_COLUMN] <= calibration_end)
        ],
        "evaluation": raw_df.loc[
            (raw_df[TIME_COLUMN] > evaluation_start)
            & (raw_df[TIME_COLUMN] <= evaluation_end)
        ],
    }
    assert segments["train"][TIME_COLUMN].max() < segments["calibration"][TIME_COLUMN].min()
    assert segments["calibration"][TIME_COLUMN].max() < segments["evaluation"][TIME_COLUMN].min()
    assert set(segments["train"][TIME_COLUMN]).isdisjoint(segments["calibration"][TIME_COLUMN])
    assert set(segments["calibration"][TIME_COLUMN]).isdisjoint(segments["evaluation"][TIME_COLUMN])
    walk_forward_segments[fold_name] = segments

    for segment_name, frame in segments.items():
        walk_forward_split_rows.append(
            {
                "fold": fold_name,
                "segment": segment_name,
                "rows": len(frame),
                "positive_samples": int(frame[TARGET].sum()),
                "positive_rate_pct": frame[TARGET].mean() * 100,
                "timestamp_groups": frame[TIME_COLUMN].nunique(),
                "start_time": frame[TIME_COLUMN].min(),
                "end_time": frame[TIME_COLUMN].max(),
            }
        )

walk_forward_split_summary = pd.DataFrame(walk_forward_split_rows).set_index(
    ["fold", "segment"]
)
display(walk_forward_split_summary)
logger.info(
    "walk_forward_split_summary=%s",
    walk_forward_split_summary.reset_index().to_dict(orient="records"),
)



def robust_stat_frame(reference_df):
    medians = reference_df.median(axis=0)
    abs_dev = reference_df.sub(medians, axis=1).abs()
    mad = abs_dev.median(axis=0)
    raw_scale = 1.4826 * mad
    scale = raw_scale.where(raw_scale > 0, 1.0).fillna(1.0)
    return pd.DataFrame({"median": medians, "mad": mad, "scale": scale})


rows  positive_samples  positive_rate_pct  \
fold   segment                                                    
fold_1 train        132137              1223           0.925555   
       calibration   43979               200           0.454763   
       evaluation    44040               326           0.740236   
fold_2 train        176116              1423           0.807990   
       calibration   44040               326           0.740236   
       evaluation    44187               152           0.343993   
fold_3 train        220156              1749           0.794437   
       calibration   44187               152           0.343993   
       evaluation    43853                39           0.088933   

                    timestamp_groups                start_time  \
fold   segment                                                   
fold_1 train                   15230 1970-06-23 03:58:55+00:00   
       calibration              1251 1970-08-18 06:51:41+00:00   
       evaluation               5415 1970-08-21 23:33:55+00:00   
fold_2 train                   16481 1970-06-23 03:58:55+00:00   
       calibration              5415 1970-08-21 23:33:55+00:00   
       evaluation               4167 1970-09-15 06:47:13+00:00   
fold_3 train                   21896 1970-06-23 03:58:55+00:00   
       calibration              4167 1970-09-15 06:47:13+00:00   
       evaluation               3186 1970-09-28 05:11:13+00:00   

                                    end_time  
fold   segment                                
fold_1 train       1970-08-18 06:51:10+00:00  
       calibration 1970-08-21 23:32:59+00:00  
       evaluation  1970-09-15 06:46:33+00:00  
fold_2 train       1970-08-21 23:32:59+00:00  
       calibration 1970-09-15 06:46:33+00:00  
       evaluation  1970-09-28 05:10:37+00:00  
fold_3 train       1970-09-15 06:46:33+00:00  
       calibration 1970-09-28 05:10:37+00:00  
       evaluation  1970-10-05 00:29:59+00:00

2026-08-25 20:07:43,011 | INFO | walk_forward_split_summary=[{'fold': 'fold_1', 'segment': 'train', 'rows': 132137, 'positive_samples': 1223, 'positive_rate_pct': 0.9255545380930399, 'timestamp_groups': 15230, 'start_time': Timestamp('1970-06-23 03:58:55+0000', tz='UTC'), 'end_time': Timestamp('1970-08-18 06:51:10+0000', tz='UTC')}, {'fold': 'fold_1', 'segment': 'calibration', 'rows': 43979, 'positive_samples': 200, 'positive_rate_pct': 0.4547625002842266, 'timestamp_groups': 1251, 'start_time': Timestamp('1970-08-18 06:51:41+0000', tz='UTC'), 'end_time': Timestamp('1970-08-21 23:32:59+0000', tz='UTC')}, {'fold': 'fold_1', 'segment': 'evaluation', 'rows': 44040, 'positive_samples': 326, 'positive_rate_pct': 0.740236148955495, 'timestamp_groups': 5415, 'start_time': Timestamp('1970-08-21 23:33:55+0000', tz='UTC'), 'end_time': Timestamp('1970-09-15 06:46:33+0000', tz='UTC')}, {'fold': 'fold_2', 'segment': 'train', 'rows': 176116, 'positive_samples': 1423, 'positive_rate_pct': 0.807990188

## 5. 센서 이상도 Fold 앙상블 학습

In [7]:
ENGINEERED_FEATURES = [
    "sensor_max_abs_robust_z",
    "sensor_mean_abs_robust_z",
    "sensor_p95_abs_robust_z",
    "sensor_abnormal_count_2",
    "sensor_abnormal_count_3",
]


def build_sensor_stats(type_train, sensor_columns):
    sensor_train = type_train[sensor_columns].astype(np.float32)
    stats = robust_stat_frame(sensor_train)
    raw_scale = 1.4826 * stats["mad"]
    return stats, int((raw_scale <= 0).sum())


def append_sensor_anomaly_features(frame, sensor_columns, stats):
    sensor_values = frame[sensor_columns].astype(np.float32)
    robust_z = sensor_values.sub(stats["median"], axis=1).div(stats["scale"], axis=1).abs()
    values = robust_z.to_numpy(dtype=np.float32, copy=False)
    engineered = pd.DataFrame(index=frame.index)
    engineered["sensor_max_abs_robust_z"] = np.nanmax(values, axis=1)
    engineered["sensor_mean_abs_robust_z"] = np.nanmean(values, axis=1)
    engineered["sensor_p95_abs_robust_z"] = np.nanpercentile(values, 95, axis=1)
    engineered["sensor_abnormal_count_2"] = (values >= 2.0).sum(axis=1).astype(np.float32)
    engineered["sensor_abnormal_count_3"] = (values >= 3.0).sum(axis=1).astype(np.float32)
    return engineered


def build_feature_frame(frame, base_feature_columns, sensor_columns, stats):
    combined = pd.concat(
        [
            frame[base_feature_columns].copy(),
            append_sensor_anomaly_features(frame, sensor_columns, stats),
        ],
        axis=1,
    )
    assert combined.index.equals(frame.index)
    assert np.isfinite(combined[ENGINEERED_FEATURES].to_numpy()).all()
    return combined


ENSEMBLE_CHECKPOINTS = [0.30, 0.40, 0.50, 0.70]
FOLD_MEMBER_CHECKPOINTS = {
    "fold_1": [0.30],
    "fold_2": [0.30, 0.40],
    "fold_3": [0.30, 0.40, 0.50],
}
FINAL_MEMBER_CHECKPOINTS = ENSEMBLE_CHECKPOINTS.copy()

prediction_targets = {}
for spec in WALK_FORWARD_SPECS:
    fold_name = spec["fold"]
    prediction_targets[f"{fold_name}_calibration"] = walk_forward_segments[fold_name]["calibration"]
    prediction_targets[f"{fold_name}_evaluation"] = walk_forward_segments[fold_name]["evaluation"]
prediction_targets["final_validation"] = validation_df

checkpoint_predictions = {
    checkpoint: {
        target_name: pd.Series(np.nan, index=frame.index, dtype="float64")
        for target_name, frame in prediction_targets.items()
        if frame[TIME_COLUMN].min() > walk_forward_boundaries[checkpoint]
    }
    for checkpoint in ENSEMBLE_CHECKPOINTS
}
ensemble_members = {checkpoint: {} for checkpoint in ENSEMBLE_CHECKPOINTS}
ensemble_training_rows = []
feature_stat_rows = []

for checkpoint in ENSEMBLE_CHECKPOINTS:
    checkpoint_train = raw_df.loc[raw_df[TIME_COLUMN] <= walk_forward_boundaries[checkpoint]]
    for inspection_type in inspection_types:
        base_feature_columns = feature_columns_by_type[inspection_type]
        sensor_columns = mapped_feature_columns_by_type[inspection_type]
        model_feature_columns = base_feature_columns + ENGINEERED_FEATURES
        type_train = checkpoint_train.loc[checkpoint_train[TYPE_COLUMN] == inspection_type]
        y_train = type_train[TARGET].astype("int8")
        assert y_train.nunique() == 2

        stats, nonpositive_scale_count = build_sensor_stats(type_train, sensor_columns)
        train_features = build_feature_frame(type_train, base_feature_columns, sensor_columns, stats)
        preprocessor = make_preprocessor(model_feature_columns)
        X_train = preprocessor.fit_transform(train_features[model_feature_columns])
        model = XGBClassifier(**XGB_PARAMS)
        model.fit(X_train, y_train, verbose=False)

        for target_name, probability_series in checkpoint_predictions[checkpoint].items():
            target_frame = prediction_targets[target_name]
            type_target = target_frame.loc[target_frame[TYPE_COLUMN] == inspection_type]
            target_features = build_feature_frame(type_target, base_feature_columns, sensor_columns, stats)
            X_target = preprocessor.transform(target_features[model_feature_columns])
            probability_series.loc[type_target.index] = model.predict_proba(X_target)[:, 1]
            del X_target, target_features

        ensemble_training_rows.append(
            {
                "checkpoint": checkpoint,
                "inspection_type": inspection_type,
                "train_rows": len(type_train),
                "train_positive": int(y_train.sum()),
                "raw_features": len(model_feature_columns),
                "engineered_features": len(ENGINEERED_FEATURES),
                "encoded_features": X_train.shape[1],
                "trees": model.get_booster().num_boosted_rounds(),
            }
        )
        feature_stat_rows.append(
            {
                "checkpoint": checkpoint,
                "inspection_type": inspection_type,
                "sensor_features": len(sensor_columns),
                "nonpositive_scale_count": nonpositive_scale_count,
            }
        )
        ensemble_members[checkpoint][inspection_type] = {
            "model": model,
            "preprocessor": preprocessor,
            "base_feature_columns": list(base_feature_columns),
            "sensor_columns": list(sensor_columns),
            "model_feature_columns": list(model_feature_columns),
            "sensor_stats": stats,
        }
        logger.info(
            "ensemble_member_fit_done checkpoint=%.2f type=%d rows=%d positive=%d engineered=sensor_anomaly",
            checkpoint,
            inspection_type,
            len(type_train),
            int(y_train.sum()),
        )
        del preprocessor, model, X_train, train_features
        gc.collect()

for checkpoint, target_map in checkpoint_predictions.items():
    for target_name, probability in target_map.items():
        assert probability.notna().all(), (checkpoint, target_name)

ensemble_training_summary = pd.DataFrame(ensemble_training_rows).set_index(["checkpoint", "inspection_type"])
feature_stats_summary = pd.DataFrame(feature_stat_rows).set_index(["checkpoint", "inspection_type"])
display(ensemble_training_summary)
display(feature_stats_summary)
logger.info("ensemble_members_trained=%d", len(ensemble_training_rows))


2026-08-25 20:07:45,714 | INFO | ensemble_member_fit_done checkpoint=0.30 type=0 rows=28277 positive=32 engineered=sensor_anomaly


2026-08-25 20:07:47,546 | INFO | ensemble_member_fit_done checkpoint=0.30 type=1 rows=22698 positive=269 engineered=sensor_anomaly


2026-08-25 20:07:51,651 | INFO | ensemble_member_fit_done checkpoint=0.30 type=2 rows=42288 positive=408 engineered=sensor_anomaly


2026-08-25 20:07:56,292 | INFO | ensemble_member_fit_done checkpoint=0.30 type=3 rows=37264 positive=510 engineered=sensor_anomaly


2026-08-25 20:07:56,527 | INFO | ensemble_member_fit_done checkpoint=0.30 type=4 rows=1610 positive=4 engineered=sensor_anomaly


...

2026-08-25 20:08:36,669 | INFO | ensemble_member_fit_done checkpoint=0.70 type=3 rows=100740 positive=648 engineered=sensor_anomaly


2026-08-25 20:08:37,143 | INFO | ensemble_member_fit_done checkpoint=0.70 type=4 rows=3813 positive=13 engineered=sensor_anomaly


train_rows  train_positive  raw_features  \
checkpoint inspection_type                                             
0.3        0                     28277              32            53   
           1                     22698             269            61   
           2                     42288             408            74   
           3                     37264             510            74   
           4                      1610               4            30   
0.4        0                     36685              43            53   
           1                     26566             289            61   
           2                     58736             500            74   
           3                     51683             583            74   
           4                      2446               8            30   
0.5        0                     43181              93            53   
           1                     29184             475            61   
           2                     77700             549            74   
           3                     67320             622            74   
           4                      2771              10            30   
0.7        0                     64273             111            53   
           1                     38900             580            61   
           2                    100470             588            74   
           3                    100740             648            74   
           4                      3813              13            30   

                            engineered_features  encoded_features  trees  
checkpoint inspection_type                                                
0.3        0                                  5                85    400  
           1                                  5               111    400  
           2                                  5               119    400  
           3                                  5               112    400  
           4                                  5                52    400  
0.4        0                                  5                87    400  
           1                                  5               115    400  
           2                                  5               119    400  
           3                                  5               112    400  
           4                                  5                52    400  
0.5        0                                  5                89    400  
           1                                  5               116    400  
           2                                  5               120    400  
           3                                  5               113    400  
           4                                  5                55    400  
0.7        0                                  5                93    400  
           1                                  5               118    400  
           2                                  5               122    400  
           3                                  5               114    400  
           4                                  5                58    400

sensor_features  nonpositive_scale_count
checkpoint inspection_type                                          
0.3        0                             44                       38
           1                             52                       42
           2                             65                       58
           3                             65                       50
           4                             21                       11
0.4        0                             44                       38
           1                             52                       41
           2                             65                       58
           3                             65                       53
           4                             21                       11
0.5        0                             44                       38
           1                             52                       41
           2                             65                       58
           3                             65                       50
           4                             21                       11
0.7        0                             44                       38
           1                             52                       42
           2                             65                       57
           3                             65                       49
           4                             21                       11

2026-08-25 20:08:37,177 | INFO | ensemble_members_trained=20


## 6. Walk-forward 미래 구간 평가

In [8]:
def mean_checkpoint_probability(checkpoints, target_name):
    probabilities = [checkpoint_predictions[checkpoint][target_name].to_numpy() for checkpoint in checkpoints]
    return pd.Series(np.mean(np.vstack(probabilities), axis=0), index=prediction_targets[target_name].index, dtype="float64")

walk_threshold_rows, walk_metric_rows, walk_type_rows = [], [], []
for spec in WALK_FORWARD_SPECS:
    fold_name = spec["fold"]
    members = FOLD_MEMBER_CHECKPOINTS[fold_name]
    result = evaluate_calibration_and_future(
        walk_forward_segments[fold_name]["calibration"], mean_checkpoint_probability(members, f"{fold_name}_calibration"),
        walk_forward_segments[fold_name]["evaluation"], mean_checkpoint_probability(members, f"{fold_name}_evaluation"), fold_name,
    )
    walk_threshold_rows.extend(result["threshold_rows"])
    walk_metric_rows.extend(result["metric_rows"])
    walk_type_rows.extend(result["type_rows"])
    logger.info("ensemble_walk_fold_done fold=%s checkpoints=%s", fold_name, members)

walk_forward_threshold_summary = pd.DataFrame(walk_threshold_rows).set_index(["stage", "scope"])
walk_forward_evaluation_metrics = pd.DataFrame(walk_metric_rows).set_index(["stage", "strategy"])
walk_forward_type_evaluation = pd.DataFrame(walk_type_rows).set_index(["stage", "inspection_type"])
walk_forward_strategy_summary = (
    walk_forward_evaluation_metrics.reset_index().groupby("strategy").agg(
        folds=("stage", "nunique"), mean_pr_auc=("pr_auc", "mean"),
        mean_recall=("recall", "mean"), min_recall=("recall", "min"),
        recall_99_folds=("recall", lambda values: int((values >= MIN_RECALL).sum())),
        mean_false_call_reduction=("false_call_reduction", "mean"),
        min_false_call_reduction=("false_call_reduction", "min"),
        total_tp=("tp", "sum"), total_fn=("fn", "sum"),
    )
)
display(walk_forward_threshold_summary[["threshold", "positive_samples", "recall", "false_call_reduction", "tp", "fn"]])
display(walk_forward_evaluation_metrics[["positive_samples", "pr_auc", "precision", "recall", "false_call_reduction", "f1", "tp", "fn", "fp", "tn"]])
display(walk_forward_type_evaluation[["threshold", "positive_samples", "pr_auc", "recall", "false_call_reduction", "tp", "fn"]])
display(walk_forward_strategy_summary)
logger.info("walk_forward_strategy_summary=%s", walk_forward_strategy_summary.to_dict(orient="index"))

2026-08-25 20:08:37,441 | INFO | ensemble_walk_fold_done fold=fold_1 checkpoints=[0.3]


2026-08-25 20:08:37,709 | INFO | ensemble_walk_fold_done fold=fold_2 checkpoints=[0.3, 0.4]


2026-08-25 20:08:37,967 | INFO | ensemble_walk_fold_done fold=fold_3 checkpoints=[0.3, 0.4, 0.5]


threshold  positive_samples    recall  false_call_reduction  \
stage  scope                                                                 
fold_1 global   0.000047               200  0.990000              0.044313   
       type_0   0.002204                11  1.000000              0.342741   
       type_1   0.001647                20  1.000000              0.467775   
       type_2   0.000026                92  1.000000              0.001773   
       type_3   0.000261                73  1.000000              0.133068   
       type_4   0.002461                 4  1.000000              0.000000   
fold_2 global   0.000782               326  0.990798              0.312188   
       type_0   0.000415                50  1.000000              0.274744   
       type_1   0.000509               186  0.994624              0.062089   
       type_2   0.000782                49  1.000000              0.355062   
       type_3   0.036720                39  1.000000              0.805103   
       type_4   0.002900                 2  1.000000              0.000000   
fold_3 global   0.000211               152  0.993421              0.197547   
       type_0   0.000356                14  1.000000              0.409430   
       type_1   0.002263                80  1.000000              0.624722   
       type_2   0.000117                32  1.000000              0.010342   
       type_3   0.000357                23  1.000000              0.335263   
       type_4   0.003162                 3  1.000000              0.000000   

                tp  fn  
stage  scope            
fold_1 global  198   2  
       type_0   11   0  
       type_1   20   0  
       type_2   92   0  
       type_3   73   0  
       type_4    4   0  
fold_2 global  323   3  
       type_0   50   0  
       type_1  185   1  
       type_2   49   0  
       type_3   39   0  
       type_4    2   0  
fold_3 global  151   1  
       type_0   14   0  
       type_1   80   0  
       type_2   32   0  
       type_3   23   0  
       type_4    3   0

positive_samples    pr_auc  precision  \
stage  strategy                                                          
fold_1 fixed_0.5                              326  0.128169   0.229358   
       global_threshold                       326  0.128169   0.007629   
       type_specific_thresholds               326  0.128169   0.008751   
fold_2 fixed_0.5                              152  0.032971   0.019417   
       global_threshold                       152  0.032971   0.005165   
       type_specific_thresholds               152  0.032971   0.006977   
fold_3 fixed_0.5                               39  0.029987   0.071429   
       global_threshold                        39  0.029987   0.001051   
       type_specific_thresholds                39  0.029987   0.001191   

                                   recall  false_call_reduction        f1  \
stage  strategy                                                             
fold_1 fixed_0.5                 0.153374              0.996157  0.183824   
       global_threshold          1.000000              0.029968  0.015143   
       type_specific_thresholds  0.978528              0.173446  0.017348   
fold_2 fixed_0.5                 0.013158              0.997706  0.015686   
       global_threshold          0.888158              0.409561  0.010271   
       type_specific_thresholds  0.822368              0.595980  0.013837   
fold_3 fixed_0.5                 0.025641              0.999703  0.037736   
       global_threshold          0.974359              0.175674  0.002100   
       type_specific_thresholds  1.000000              0.253458  0.002379   

                                  tp   fn     fp     tn  
stage  strategy                                          
fold_1 fixed_0.5                  50  276    168  43546  
       global_threshold          326    0  42404   1310  
       type_specific_thresholds  319    7  36132   7582  
fold_2 fixed_0.5                   2  150    101  43934  
       global_threshold          135   17  26000  18035  
       type_specific_thresholds  125   27  17791  26244  
fold_3 fixed_0.5                   1   38     13  43801  
       global_threshold           38    1  36117   7697  
       type_specific_thresholds   39    0  32709  11105

threshold  positive_samples    pr_auc    recall  \
stage  inspection_type                                                    
fold_1 0                 0.002204                50  0.021614  0.940000   
       1                 0.001647               186  0.188273  0.978495   
       2                 0.000026                49  0.342835  1.000000   
       3                 0.000261                39  0.480584  1.000000   
       4                 0.002461                 2  0.006154  1.000000   
fold_2 0                 0.000415                14  0.004199  0.928571   
       1                 0.000509                80  0.269400  1.000000   
       2                 0.000782                32  0.033841  0.812500   
       3                 0.036720                23  0.001916  0.130435   
       4                 0.002900                 3  0.004298  1.000000   
fold_3 0                 0.000356                 4  0.006581  1.000000   
       1                 0.002263                25  0.025637  1.000000   
       2                 0.000117                 7  0.054640  1.000000   
       3                 0.000357                 3  0.170470  1.000000   
       4                 0.003162                 0       NaN       NaN   

                        false_call_reduction   tp  fn  
stage  inspection_type                                 
fold_1 0                            0.638225   47   3  
       1                            0.333059  182   4  
       2                            0.005234   49   0  
       3                            0.164059   39   0  
       4                            0.000000    2   0  
fold_2 0                            0.414224   13   1  
       1                            0.291726   80   0  
       2                            0.268329   26   6  
       3                            0.904796    3  20  
       4                            0.000000    3   0  
fold_3 0                            0.221102    4   0  
       1                            0.293702   25   0  
       2                            0.032861    7   0  
       3                            0.520679    3   0  
       4                            0.000000    0   0

,folds,mean_pr_auc,mean_recall,min_recall,recall_99_folds,mean_false_call_reduction,min_false_call_reduction,total_tp,total_fn
strategy,,,,,,,,,
fixed_0.5,3,0.063709,0.064058,0.013158,0,0.997855,0.996157,53,464
global_threshold,3,0.063709,0.954172,0.888158,1,0.205068,0.029968,499,18
type_specific_thresholds,3,0.063709,0.933632,0.822368,1,0.340961,0.173446,483,34


2026-08-25 20:08:37,987 | INFO | walk_forward_strategy_summary={'fixed_0.5': {'folds': 3, 'mean_pr_auc': 0.06370877279756072, 'mean_recall': 0.06405771783556737, 'min_recall': 0.013157894736842105, 'recall_99_folds': 0, 'mean_false_call_reduction': 0.9978554995816212, 'min_false_call_reduction': 0.9961568376263897, 'total_tp': 53, 'total_fn': 464}, 'global_threshold': {'folds': 3, 'mean_pr_auc': 0.06370877279756072, 'mean_recall': 0.9541722896986055, 'min_recall': 0.8881578947368421, 'recall_99_folds': 1, 'mean_false_call_reduction': 0.20506751163354853, 'min_false_call_reduction': 0.02996751612755639, 'total_tp': 499, 'total_fn': 18}, 'type_specific_thresholds': {'folds': 3, 'mean_pr_auc': 0.06370877279756072, 'mean_recall': 0.9336320094715317, 'min_recall': 0.8223684210526315, 'recall_99_folds': 1, 'mean_false_call_reduction': 0.3409612823444543, 'min_false_call_reduction': 0.1734455780756737, 'total_tp': 483, 'total_fn': 34}}


## 7. 최종 Validation 비교

In [9]:
validation_probability = mean_checkpoint_probability(
    FINAL_MEMBER_CHECKPOINTS,
    "final_validation",
)
model_summary = pd.Series(
    {
        "ensemble_members": len(FINAL_MEMBER_CHECKPOINTS),
        "member_checkpoints": FINAL_MEMBER_CHECKPOINTS,
        "engineered_features": ENGINEERED_FEATURES,
    },
    name="final_ensemble",
)

global_threshold_selection = select_threshold(
    validation_df[TARGET],
    validation_probability,
    min_recall=MIN_RECALL,
)
thresholds_by_type = {}
type_threshold_rows = []
type_validation_prediction = pd.Series(np.nan, index=validation_df.index, dtype="float64")

for inspection_type in inspection_types:
    type_validation = validation_df.loc[validation_df[TYPE_COLUMN] == inspection_type]
    type_probability = validation_probability.loc[type_validation.index]
    selection = select_threshold(type_validation[TARGET], type_probability, min_recall=MIN_RECALL)
    thresholds_by_type[inspection_type] = selection["threshold"]
    type_threshold_rows.append({"inspection_type": inspection_type, **selection})
    type_validation_prediction.loc[type_validation.index] = (
        type_probability >= selection["threshold"]
    ).astype("int8")

fixed_validation_metrics = pd.Series(
    evaluate_probabilities(validation_df[TARGET], validation_probability, DECISION_THRESHOLD),
    name="fixed_0.5",
)
global_validation_metrics = pd.Series(
    evaluate_probabilities(
        validation_df[TARGET],
        validation_probability,
        global_threshold_selection["threshold"],
    ),
    name="global_threshold",
)
type_specific_validation_metrics = pd.Series(
    evaluate_predictions(
        validation_df[TARGET],
        type_validation_prediction,
        validation_probability,
    ),
    name="type_specific_thresholds",
)
validation_strategy_metrics = pd.DataFrame(
    {
        "fixed_0.5": fixed_validation_metrics,
        "global_threshold": global_validation_metrics,
        "type_specific_thresholds": type_specific_validation_metrics,
    }
).T
threshold_summary = pd.concat(
    [
        pd.DataFrame([{"scope": "global", **global_threshold_selection}]).set_index("scope"),
        pd.DataFrame(type_threshold_rows).set_index("inspection_type").rename_axis("scope"),
    ]
)

display(model_summary)
display(threshold_summary[["threshold", "positive_samples", "recall", "false_call_reduction", "tp", "fn", "fp", "tn"]])
display(validation_strategy_metrics[["pr_auc", "precision", "recall", "false_call_reduction", "f1", "tp", "fn", "fp", "tn"]])
logger.info("validation_strategy_metrics=%s", validation_strategy_metrics.to_dict(orient="index"))


ensemble_members                                                       4
member_checkpoints                                  [0.3, 0.4, 0.5, 0.7]
engineered_features    [sensor_max_abs_robust_z, sensor_mean_abs_robu...
Name: final_ensemble, dtype: object

,threshold,positive_samples,recall,false_call_reduction,tp,fn,fp,tn
scope,,,,,,,,
global,0.000637,357,0.991597,0.529208,354,3,20559,23110
0,0.000496,12,1.000000,0.478346,12,0,6926,6351
1,0.001165,224,0.991071,0.485479,222,2,3189,3009
2,0.000393,27,1.000000,0.435100,27,0,4030,3104
3,0.002965,21,1.000000,0.847637,21,0,2473,13758
4,0.003233,73,1.000000,0.000000,73,0,829,0


,pr_auc,precision,recall,false_call_reduction,f1,tp,fn,fp,tn
fixed_0.5,0.389631,0.681818,0.168067,0.999359,0.269663,60.0,297.0,28.0,43641.0
global_threshold,0.389631,0.016927,0.991597,0.529208,0.033286,354.0,3.0,20559.0,23110.0
type_specific_thresholds,0.389631,0.019942,0.994398,0.600472,0.039099,355.0,2.0,17447.0,26222.0


2026-08-25 20:08:38,211 | INFO | validation_strategy_metrics={'fixed_0.5': {'rows': 44026.0, 'positive_samples': 357.0, 'tn': 43641.0, 'fp': 28.0, 'fn': 297.0, 'tp': 60.0, 'accuracy': 0.9926179984554582, 'precision': 0.6818181818181818, 'recall': 0.16806722689075632, 'false_call_reduction': 0.999358812887861, 'f1': 0.2696629213483146, 'roc_auc': 0.936500634740603, 'pr_auc': 0.38963102734060207}, 'global_threshold': {'rows': 44026.0, 'positive_samples': 357.0, 'tn': 23110.0, 'fp': 20559.0, 'fn': 3.0, 'tp': 354.0, 'accuracy': 0.5329577976650162, 'precision': 0.016927270119064697, 'recall': 0.9915966386554622, 'false_call_reduction': 0.5292083629119054, 'f1': 0.033286318758815235, 'roc_auc': 0.936500634740603, 'pr_auc': 0.38963102734060207}, 'type_specific_thresholds': {'rows': 44026.0, 'positive_samples': 357.0, 'tn': 26222.0, 'fp': 17447.0, 'fn': 2.0, 'tp': 355.0, 'accuracy': 0.603666015536274, 'precision': 0.019941579597798, 'recall': 0.9943977591036415, 'false_call_reduction': 0.60047

## 8. 원본 무결성과 결과 요약

In [10]:
DATA_SHA256_AFTER = sha256_file(DATA_PATH)
MAPPING_SHA256_AFTER = sha256_file(MAPPING_PATH)
assert DATA_SHA256_AFTER == DATA_SHA256_BEFORE
assert MAPPING_SHA256_AFTER == MAPPING_SHA256_BEFORE

verification = pd.Series(
    {
        "dataset_sha256_unchanged": True,
        "mapping_sha256_unchanged": True,
        "trained_model_units": len(FINAL_MEMBER_CHECKPOINTS) * len(inspection_types),
        "final_ensemble_members": len(FINAL_MEMBER_CHECKPOINTS),
        "test_holdout_used_for_selection": False,
        "test_holdout_predicted": False,
        "global_threshold": global_threshold_selection["threshold"],
        "type_thresholds": thresholds_by_type,
        "log_file": f"docs/peace/{LOG_PATH.name}",
    },
    name="verification",
)
display(verification)

result_summary = {
    "experiment_id": EXPERIMENT_ID,
    "notebook_path": f"notebooks/{EXPERIMENT_ID}.ipynb",
    "log_path": f"docs/peace/{LOG_PATH.name}",
    "evaluation_scope": "walk_forward_and_validation_only",
    "baseline_005_reference": {
        "walk_forward": BASELINE_005_WALK,
        "validation": BASELINE_005_VALIDATION,
    },
    "engineered_features": ENGINEERED_FEATURES,
    "ensemble_checkpoints": ENSEMBLE_CHECKPOINTS,
    "fold_member_checkpoints": FOLD_MEMBER_CHECKPOINTS,
    "walk_forward_strategy_summary": to_builtin_dict(
        walk_forward_strategy_summary.to_dict(orient="index")
    ),
    "validation_strategy_metrics": to_builtin_dict(
        validation_strategy_metrics.to_dict(orient="index")
    ),
    "threshold_summary": to_builtin_dict(
        threshold_summary.reset_index().to_dict(orient="records")
    ),
}
logger.info("result_summary_json=%s", json.dumps(result_summary, ensure_ascii=False, sort_keys=True))
logger.info("source_integrity=PASS test_holdout_predicted=False")
logger.info("experiment_complete=%s", EXPERIMENT_ID)
for handler in logger.handlers:
    handler.flush()

baseline_walk = BASELINE_005_WALK["global_threshold"]
baseline_validation = BASELINE_005_VALIDATION["global_threshold"]
walk = walk_forward_strategy_summary.loc["global_threshold"]
validation = validation_strategy_metrics.loc["global_threshold"]

print("005 대비 Walk-forward 평균 Recall 변화:", float(walk["mean_recall"] - baseline_walk["mean_recall"]))
print("005 대비 Walk-forward 평균 FCR 변화:", float(walk["mean_false_call_reduction"] - baseline_walk["mean_false_call_reduction"]))
print("005 대비 Validation PR-AUC 변화:", float(validation["pr_auc"] - baseline_validation["pr_auc"]))
print("005 대비 Validation FCR 변화:", float(validation["false_call_reduction"] - baseline_validation["false_call_reduction"]))


dataset_sha256_unchanged                                                        True
mapping_sha256_unchanged                                                        True
trained_model_units                                                               20
final_ensemble_members                                                             4
test_holdout_used_for_selection                                                False
test_holdout_predicted                                                         False
global_threshold                                                            0.000637
type_thresholds                    {0: 0.0004959957550454419, 1: 0.00116534719563...
log_file                           docs/peace/0825_peace_018_type_expert_fold_ens...
Name: verification, dtype: object

2026-08-25 20:08:38,410 | INFO | result_summary_json={"baseline_005_reference": {"validation": {"global_threshold": {"false_call_reduction": 0.6110742174082301, "fn": 3, "fp": 16984, "pr_auc": 0.3826504981185914, "recall": 0.9915966386554622, "tn": 26685, "tp": 354}}, "walk_forward": {"global_threshold": {"mean_false_call_reduction": 0.17635946579785103, "mean_pr_auc": 0.06666848491626425, "mean_recall": 0.9868421052631579, "min_recall": 0.9605263157894737, "recall_99_folds": 2}}}, "engineered_features": ["sensor_max_abs_robust_z", "sensor_mean_abs_robust_z", "sensor_p95_abs_robust_z", "sensor_abnormal_count_2", "sensor_abnormal_count_3"], "ensemble_checkpoints": [0.3, 0.4, 0.5, 0.7], "evaluation_scope": "walk_forward_and_validation_only", "experiment_id": "0825_peace_018_type_expert_fold_ensemble_sensor_anomaly", "fold_member_checkpoints": {"fold_1": [0.3], "fold_2": [0.3, 0.4], "fold_3": [0.3, 0.4, 0.5]}, "log_path": "docs/peace/0825_peace_018_type_expert_fold_ensemble_sensor_anomaly

2026-08-25 20:08:38,410 | INFO | source_integrity=PASS test_holdout_predicted=False


2026-08-25 20:08:38,411 | INFO | experiment_complete=0825_peace_018_type_expert_fold_ensemble_sensor_anomaly


005 대비 Walk-forward 평균 Recall 변화: -0.032669815564552396
005 대비 Walk-forward 평균 FCR 변화: 0.0287080458356975
005 대비 Validation PR-AUC 변화: 0.006980529222010645
005 대비 Validation FCR 변화: -0.08186585449632466
